# Exploração de modelo — propensity do X-learner

Notebook de **exploração** (sem testes): dois experimentos sobre o `model_p`
estimado do X-learner (`src.uplift`, propensity por X desde 2026-07-16).

1. **Histograma da propensity** prevista por linha — quanto `g(x)` de fato varia.
   Se degenerar perto de uma constante, estimar propensity mal muda a ordenação;
   se espalha por [0, 1], confirma que ela injeta sinal de autosseleção no score.
2. **X-learner com propensity clipada em [0.1, 0.9]** — mesmo modelo ajustado,
   só o peso `p` truncado nas caudas, contra a versão sem clip. Compara o Qini AUC.


In [1]:
import os, sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import gc
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

import numpy as np
import pandas as pd

from src import split, uplift, uplift_eval, viz
from src.uplift import _design_matrix, OFFER_TYPE_COLUMN, TREATMENT_COLUMN, OUTCOME_COLUMN
from src.models import UpliftModel
from src.config import load
from src.pipeline import build_spark

pd.set_option("display.max_columns", None)
viz.setup_theme()

cfg = load()

/home/caioolubini/Projects/ifood-coupons-uplift/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Dado — mesmo split temporal e exclusão de informational do notebook 2

In [2]:
spark = build_spark(cfg, app_name="ifood-uplift-model-exploration")
processed = spark.read.parquet(str(cfg.processed_dir))

train_sdf, holdout_sdf = split.temporal_split(processed, cfg)
split.assert_temporal_order(train_sdf, holdout_sdf)
train_sdf = split.exclude_informational(train_sdf)
holdout_sdf = split.exclude_informational(holdout_sdf)

train_df = train_sdf.toPandas()
holdout_df = holdout_sdf.toPandas()

# Free the JVM before causalml/LGBM allocate.
spark.stop()
del processed, train_sdf, holdout_sdf
gc.collect()

print(f"train:   {len(train_df):,} rows")
print(f"holdout: {len(holdout_df):,} rows")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/07/16 15:07:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/07/16 15:07:35 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


train:   40,630 rows
holdout: 20,412 rows


## Ajuste — X-learner de produção (propensity estimada por X)

In [3]:
# NOTA: a produção (`src.uplift`) usa propensity FIXA (revertida em 2026-07-16).
# Este apêndice estuda a variante de propensity ESTIMADA, então ajusta aqui um
# X-learner com `model_p` (LGBMClassifier) próprio — sem depender de `src` expor um.
from causalml.inference.meta import BaseXRegressor
from lightgbm import LGBMClassifier, LGBMRegressor
from src.uplift import _design_matrix, _XLEARNER_FEATURES

def _learner():
    return LGBMRegressor(n_estimators=cfg.xlearner_n_estimators, max_depth=cfg.xlearner_max_depth,
                         learning_rate=cfg.xlearner_learning_rate, random_state=cfg.seed, verbose=-1)

def _propensity_learner():
    return LGBMClassifier(n_estimators=cfg.xlearner_n_estimators, max_depth=cfg.xlearner_max_depth,
                          learning_rate=cfg.xlearner_learning_rate, random_state=cfg.seed, verbose=-1)

models = {}
for offer_type, group in train_df.groupby(OFFER_TYPE_COLUMN):
    X = _design_matrix(group)
    m = BaseXRegressor(learner=_learner(), control_name=0)
    m.model_p = _propensity_learner()  # estima g(x) por X, tolera nulos de G8
    m.fit(X=X, treatment=group[TREATMENT_COLUMN].to_numpy(), y=group[OUTCOME_COLUMN].to_numpy())
    models[offer_type] = m  # cada BaseXRegressor guarda seu model_p ajustado


## 1. Histograma da propensity prevista

`model_p.predict_proba(X)[:, 1]` por linha do holdout, por `offer_type`. Quanto
`g(x)` se afasta da taxa de view constante que a versão fixa usava.

In [4]:
prop_rows = []
for offer_type, group in holdout_df.groupby(OFFER_TYPE_COLUMN):
    model = models[offer_type]
    X = _design_matrix(group)
    p = model.model_p.predict_proba(X)[:, 1]
    prop_rows.append(pd.DataFrame({
        "offer_type": offer_type,
        "propensity": p,
        "view_rate_observada": group[TREATMENT_COLUMN].mean(),
    }))

prop = pd.concat(prop_rows, ignore_index=True)
prop.groupby("offer_type")["propensity"].describe()[["mean", "std", "min", "25%", "50%", "75%", "max"]]

,mean,std,min,25%,50%,75%,max
offer_type,,,,,,,
bogo,0.761096,0.217505,0.052532,0.675364,0.851763,0.919638,0.992365
discount,0.626045,0.291157,0.034596,0.359943,0.737425,0.876970,0.989521


In [5]:
# Linha de referência: a taxa de view observada (o que a propensity FIXA usava).
view_rate_geral = float(holdout_df[TREATMENT_COLUMN].mean())

viz.histogram(
    prop, x="propensity",
    title=f"Propensity estimada g(x) espalha em torno da taxa de view ({view_rate_geral:.2f})",
    subtitle="P(viu | X) por linha do holdout, LGBM model_p do X-learner. Linha = taxa de view observada.",
    xlabel="propensity estimada g(x)",
    markers=[(view_rate_geral, "taxa de view")],
)

In [6]:
# Uma leitura por offer_type (as caudas [0.1, 0.9] são o alvo do clip do passo 2).
for offer_type, g in prop.groupby("offer_type"):
    vr = float(g["view_rate_observada"].iloc[0])
    fora = ((g["propensity"] < 0.1) | (g["propensity"] > 0.9)).mean()
    viz.histogram(
        g, x="propensity",
        title=f"{offer_type} — g(x) vs. taxa de view {vr:.2f}  ({fora:.1%} fora de [0.1, 0.9])",
        subtitle="Barras verticais: caudas 0.1 e 0.9 (onde o clip do passo 2 atua) e a taxa de view.",
        xlabel="propensity estimada g(x)",
        markers=[(vr, "taxa de view"), (0.1, "clip lo"), (0.9, "clip hi")],
    ).show()

## 2. X-learner com propensity clipada em [0.1, 0.9]

**Mesmo** X-learner ajustado — não re-treina nada. A fórmula do X-learner pesa
os dois estimadores de CATE por `g(x)`:
`τ(x) = g(x)·τ_c(x) + (1−g(x))·τ_t(x)`. Passar `p` explícito a `model.predict`
sobrescreve o `model_p` interno só na **predição**, então clipar `p` nas caudas
testa se propensities extremas (peso quase todo num único estimador) estavam
degradando a ordenação — sem tocar em `src/`.

In [7]:
def predict_uplift_clipped(models, df, clip):
    """Uplift por linha reusando o X-learner ajustado, mas com a propensity
    prevista clipada em `clip=(lo, hi)` antes de entrar na combinação de CATE.
    clip=None reproduz a predição de produção (model_p sem clip)."""
    out = pd.Series(index=df.index, dtype=float)
    for offer_type, group in df.groupby(OFFER_TYPE_COLUMN):
        model = models[offer_type]
        X = _design_matrix(group)
        if clip is None:
            te = model.predict(X=X, verbose=False)
        else:
            p = model.model_p.predict_proba(X)[:, 1]
            p = np.clip(p, clip[0], clip[1])
            te = model.predict(X=X, p=p, verbose=False)
        out.loc[group.index] = te[:, 0] if te.ndim == 2 else te.ravel()
    return out

In [8]:
uplift_base = predict_uplift_clipped(models, holdout_df, clip=None)      # produção (sem clip)
uplift_clip = predict_uplift_clipped(models, holdout_df, clip=(0.1, 0.9)) # clipada

qini_base = uplift_eval.qini(holdout_df["converted"], uplift_base, holdout_df["treatment"])
qini_clip = uplift_eval.qini(holdout_df["converted"], uplift_clip, holdout_df["treatment"])

pd.DataFrame({
    "variante": ["propensity estimada (produção)", "propensity clipada [0.1, 0.9]"],
    "qini_auc": [qini_base, qini_clip],
    "corr_de_rank_com_base": [1.0, float(pd.Series(uplift_base).corr(pd.Series(uplift_clip), method="spearman"))],
})

,variante,qini_auc,corr_de_rank_com_base
0,propensity estimada (produção),0.013207,1.000000
1,"propensity clipada [0.1, 0.9]",0.017981,0.971313


In [9]:
# Curvas Qini lado a lado.
curva_base = uplift_eval.qini_curve_points(holdout_df["converted"], uplift_base, holdout_df["treatment"])
curva_clip = uplift_eval.qini_curve_points(holdout_df["converted"], uplift_clip, holdout_df["treatment"])

fig = viz.figure(
    title=f"Clip [0.1, 0.9] muda o Qini de {qini_base:.3f} para {qini_clip:.3f}",
    subtitle="Ganho incremental acumulado no holdout, ordenado por uplift previsto — sem clip vs. clipado.",
)
import plotly.graph_objects as go
pal = viz.palette("light")
fig.add_trace(go.Scatter(x=curva_base["n_treated"], y=curva_base["gain"],
                         mode="lines", name="estimada (produção)", line=dict(color=pal[0], width=2.5)))
fig.add_trace(go.Scatter(x=curva_clip["n_treated"], y=curva_clip["gain"],
                         mode="lines", name="clipada [0.1, 0.9]", line=dict(color=pal[1], width=2.5)))
fig.add_trace(go.Scatter(
    x=[curva_base["n_treated"].iloc[0], curva_base["n_treated"].iloc[-1]],
    y=[curva_base["gain"].iloc[0], curva_base["gain"].iloc[-1]],
    mode="lines", name="random", line=dict(color=pal[0], width=1.5, dash="dot")))
fig.update_layout(xaxis_title="clientes tratados", yaxis_title="ganho incremental acumulado")
fig

## 3. Overlap de feature importance — propensity vs. uplift

A pergunta: **o `model_p` e o modelo de uplift priorizam as mesmas features?**
Se sim, é evidência direta de que a propensity estimada está aprendendo o mesmo
padrão *preditivo* (quem-vê ≈ quem-compra) que polui a ordenação *causal* — o
peso `g(x)` na fórmula do X-learner injeta exatamente o sinal que o Qini penaliza.

Para ser apples-to-apples, mede-se **importância por permutação** nos dois lados,
sobre o **mesmo espaço de features** (`_XLEARNER_FEATURES`) e a mesma matriz de
design, agregada por média ponderada por linhas e normalizada para somar 1:

- **uplift**: `uplift.causal_importance` — permuta sobre o meta-modelo `X → τ̂(X)`
  (o que dirige o *efeito*).
- **propensity**: permutação sobre o `model_p` já ajustado (log-loss de `treatment`)
  — o que dirige *quem vê*.

In [10]:
from sklearn.inspection import permutation_importance
from src.uplift import _XLEARNER_FEATURES

# --- Importância causal do modelo de uplift (reusa a função de src, sem reimplementar) ---
imp_uplift = uplift.causal_importance(models, holdout_df, cfg)

# --- Importância por permutação do model_p, no MESMO molde (permutação, ponderado por linhas, normalizado) ---
def propensity_permutation_importance(models, df, cfg):
    features = _XLEARNER_FEATURES
    total = pd.Series(0.0, index=features)
    n_total = 0
    for offer_type, group in df.groupby(OFFER_TYPE_COLUMN):
        model_p = models[offer_type].model_p
        X = _design_matrix(group)
        y = group[TREATMENT_COLUMN].to_numpy()
        r = permutation_importance(
            model_p, X, y,
            scoring="neg_log_loss", n_repeats=10, random_state=cfg.seed,
        )
        s = pd.Series(r.importances_mean, index=features).clip(lower=0.0)
        total = total + s * len(group)
        n_total += len(group)
    mean_imp = total / n_total
    return (mean_imp / mean_imp.sum()).sort_values(ascending=False)

imp_prop = propensity_permutation_importance(models, holdout_df, cfg)

overlap = pd.DataFrame({
    "imp_propensity": imp_prop,
    "imp_uplift": imp_uplift,
}).fillna(0.0)
overlap["rank_propensity"] = overlap["imp_propensity"].rank(ascending=False).astype(int)
overlap["rank_uplift"] = overlap["imp_uplift"].rank(ascending=False).astype(int)
overlap.sort_values("imp_propensity", ascending=False)

,imp_propensity,imp_uplift,rank_propensity,rank_uplift
n_channels,0.437045,0.019608,1,13
channel_social,0.197410,0.002137,2,26
hist_avg_ticket,0.090603,0.079335,3,5
credit_card_limit,0.049789,0.133842,4,1
hist_view_rate,0.040272,0.012153,5,16
discount_value,0.027790,0.014222,6,15
duration,0.025703,0.008012,7,18
hist_offers_received_discount,0.024808,0.100374,8,4
age,0.024349,0.069610,9,6
hist_offers_received_bogo,0.023691,0.004126,10,21


### Concordância global

Duas leituras: **Spearman** (as ordenações completas concordam?) e a **sobreposição
do top-k** (as features mais importantes são as mesmas?).

In [11]:
from scipy.stats import spearmanr

rho, pval = spearmanr(overlap["imp_propensity"], overlap["imp_uplift"])

def jaccard_topk(a: pd.Series, b: pd.Series, k: int) -> float:
    ta, tb = set(a.nlargest(k).index), set(b.nlargest(k).index)
    return len(ta & tb) / len(ta | tb)

linhas = []
for k in (3, 5, 8):
    top_p = set(imp_prop.nlargest(k).index)
    top_u = set(imp_uplift.nlargest(k).index)
    linhas.append({
        "k": k,
        "jaccard": jaccard_topk(imp_prop, imp_uplift, k),
        "compartilhadas": ", ".join(sorted(top_p & top_u)) or "—",
    })

print(f"Spearman(importância propensity, importância uplift) = {rho:.3f}  (p={pval:.3f})")
pd.DataFrame(linhas)

Spearman(importância propensity, importância uplift) = 0.561  (p=0.001)


,k,jaccard,compartilhadas
0,3,0.000000,—
1,5,0.250000,"credit_card_limit, hist_avg_ticket"
2,8,0.230769,"credit_card_limit, hist_avg_ticket, hist_offer..."


### Comparação visual — importância nas duas óticas, mesma feature lado a lado

In [12]:
import plotly.graph_objects as go
pal = viz.palette("light")

# Ordena pelas mais importantes na UNIÃO dos dois, top-12 para caber.
ordem = (overlap["imp_propensity"] + overlap["imp_uplift"]).nlargest(12).index[::-1]
sub = overlap.loc[ordem]

fig = viz.figure(
    title=f"Propensity e uplift priorizam features diferentes (Spearman = {rho:.2f})"
          if rho < 0.5 else
          f"Propensity e uplift priorizam as MESMAS features (Spearman = {rho:.2f})",
    subtitle="Importância por permutação, normalizada (soma 1), top-12 features pela soma das duas óticas.",
)
fig.add_trace(go.Bar(y=sub.index, x=sub["imp_propensity"], name="propensity (quem vê)",
                     orientation="h", marker_color=pal[0]))
fig.add_trace(go.Bar(y=sub.index, x=sub["imp_uplift"], name="uplift (efeito causal)",
                     orientation="h", marker_color=pal[1]))
fig.update_layout(barmode="group", xaxis_title="importância normalizada", yaxis_title=None)
fig

In [13]:
# Scatter: cada feature um ponto. Diagonal = mesma importância nos dois.
# Pontos longe da diagonal = feature que um modelo usa e o outro ignora.
fig = viz.figure(
    title="Divergência feature a feature — longe da diagonal = padrão aprendido por só um modelo",
    subtitle="Cada ponto é uma feature. Eixo x: importância na propensity; eixo y: no uplift.",
)
lim = float(max(overlap["imp_propensity"].max(), overlap["imp_uplift"].max())) * 1.1
fig.add_trace(go.Scatter(x=[0, lim], y=[0, lim], mode="lines",
                         line=dict(color=viz.ink("light")[0], width=1, dash="dot"),
                         name="igual importância", hoverinfo="skip"))
fig.add_trace(go.Scatter(
    x=overlap["imp_propensity"], y=overlap["imp_uplift"], mode="markers+text",
    text=overlap.index, textposition="top center", textfont=dict(size=9),
    marker=dict(color=pal[1], size=9), name="feature",
    hovertemplate="%{text}<br>propensity=%{x:.3f}<br>uplift=%{y:.3f}<extra></extra>",
))
fig.update_layout(xaxis_title="importância na propensity", yaxis_title="importância no uplift",
                  xaxis_range=[0, lim], yaxis_range=[0, lim], showlegend=False)
fig

## 4. Variante — propensity treinada sem as features que os dois modelos compartilham

Hipótese do passo 3: a propensity contamina a ordenação causal porque `g(x)`
carrega uma faixa de features (`credit_card_limit`, `hist_avg_ticket`,
`hist_offers_received_discount`…) que **também** dirigem o uplift. Se removermos
essas features do treino do `model_p`, ele fica com o eixo que é só dele
(canal/entrega) e o peso `g(x)` deveria contaminar menos a ordenação do X-learner.

**Critério de "compartilhada"**: feature acima da mediana de importância nos
**dois** modelos ao mesmo tempo (canto superior-direito do scatter do passo 3) —
data-driven, sem lista chumbada. O X-learner (estimadores de CATE) é o **mesmo**;
só o `model_p` é re-treinado num subconjunto de features, e o `p` resultante entra
via `model.predict(p=...)`, como no passo 2.

In [14]:
from lightgbm import LGBMClassifier

# Features "compartilhadas" = acima da mediana de importância nos DOIS modelos.
med_p = overlap["imp_propensity"][overlap["imp_propensity"] > 0].median()
med_u = overlap["imp_uplift"][overlap["imp_uplift"] > 0].median()
shared = overlap[(overlap["imp_propensity"] >= med_p) & (overlap["imp_uplift"] >= med_u)].index.tolist()

kept = [f for f in _XLEARNER_FEATURES if f not in shared]
print(f"Removidas do model_p ({len(shared)}): {shared}")
print(f"Mantidas no model_p ({len(kept)}):  {kept}")

Removidas do model_p (9): ['age', 'credit_card_limit', 'discount_value', 'hist_avg_ticket', 'hist_frequency', 'hist_offers_received_discount', 'hist_offers_viewed', 'n_channels', 'tenure_days']
Mantidas no model_p (24):  ['gender', 'identity_missing', 'hist_spend_total', 'hist_txn_count', 'hist_spend_std', 'hist_recency_days', 'hist_spend_trend', 'hist_offers_received', 'hist_offers_received_bogo', 'hist_offers_received_info', 'hist_offers_completed', 'hist_view_rate', 'hist_conv_rate_bogo', 'hist_conv_rate_discount', 'hist_completed_unseen_flag', 'hist_time_view_to_conv', 'min_value', 'duration', 'channel_web', 'channel_email', 'channel_mobile', 'channel_social', 'discount_to_minvalue_ratio', 'n_concurrent_offers']


In [15]:
def propensity_scores_reduced(models, df, kept_features, cfg):
    """Re-treina só o model_p de cada offer_type num subconjunto de features
    (mesmos hiperparâmetros de _make_propensity_learner) e devolve p por linha,
    na ordem de df. Os estimadores de CATE do X-learner ficam intactos."""
    p_out = pd.Series(index=df.index, dtype=float)
    for offer_type, group in df.groupby(OFFER_TYPE_COLUMN):
        X_full = _design_matrix(group)
        X_red = X_full[kept_features]
        clf = LGBMClassifier(
            n_estimators=cfg.xlearner_n_estimators,
            max_depth=cfg.xlearner_max_depth,
            learning_rate=cfg.xlearner_learning_rate,
            random_state=cfg.seed, verbose=-1,
        )
        clf.fit(X_red, group[TREATMENT_COLUMN].to_numpy())
        p_out.loc[group.index] = clf.predict_proba(X_red)[:, 1]
    return p_out

def predict_uplift_with_p(models, df, p_series):
    """Uplift por linha reusando o X-learner ajustado, com propensity externa p_series."""
    out = pd.Series(index=df.index, dtype=float)
    for offer_type, group in df.groupby(OFFER_TYPE_COLUMN):
        model = models[offer_type]
        X = _design_matrix(group)
        p = p_series.loc[group.index].to_numpy()
        te = model.predict(X=X, p=p, verbose=False)
        out.loc[group.index] = te[:, 0] if te.ndim == 2 else te.ravel()
    return out

# NOTA metodológica: o model_p reduzido é ajustado no próprio holdout (exploração,
# não produção) — coerente com o passo 3, que também mede importância no holdout.
p_reduced = propensity_scores_reduced(models, holdout_df, kept, cfg)
uplift_reduced = predict_uplift_with_p(models, holdout_df, p_reduced)
qini_reduced = uplift_eval.qini(holdout_df["converted"], uplift_reduced, holdout_df["treatment"])
print(f"Qini AUC (propensity sem features compartilhadas) = {qini_reduced:.4f}")

Qini AUC (propensity sem features compartilhadas) = 0.0147


### Placar final — as quatro variantes de propensity, mesmo X-learner

In [16]:
placar = pd.DataFrame({
    "variante": [
        "estimada — todas as features (produção)",
        "estimada — clipada [0.1, 0.9]",
        "estimada — sem features compartilhadas",
    ],
    "qini_auc": [qini_base, qini_clip, qini_reduced],
    "corr_rank_com_base": [
        1.0,
        float(pd.Series(uplift_base).corr(pd.Series(uplift_clip), method="spearman")),
        float(pd.Series(uplift_base).corr(pd.Series(uplift_reduced), method="spearman")),
    ],
})
placar

,variante,qini_auc,corr_rank_com_base
0,estimada — todas as features (produção),0.013207,1.000000
1,"estimada — clipada [0.1, 0.9]",0.017981,0.971313
2,estimada — sem features compartilhadas,0.014732,0.960006


In [17]:
import plotly.graph_objects as go
pal = viz.palette("light")

curva_red = uplift_eval.qini_curve_points(holdout_df["converted"], uplift_reduced, holdout_df["treatment"])

fig = viz.figure(
    title="Qini das três variantes de propensity sobre o mesmo X-learner",
    subtitle="Ganho incremental acumulado no holdout, ordenado por uplift previsto.",
)
series = [
    (curva_base, f"estimada / todas ({qini_base:.3f})", pal[0]),
    (curva_clip, f"clipada [0.1,0.9] ({qini_clip:.3f})", pal[1]),
    (curva_red,  f"sem compartilhadas ({qini_reduced:.3f})", pal[2]),
]
for curva, nome, cor in series:
    fig.add_trace(go.Scatter(x=curva["n_treated"], y=curva["gain"], mode="lines",
                             name=nome, line=dict(color=cor, width=2.5)))
fig.add_trace(go.Scatter(
    x=[curva_base["n_treated"].iloc[0], curva_base["n_treated"].iloc[-1]],
    y=[curva_base["gain"].iloc[0], curva_base["gain"].iloc[-1]],
    mode="lines", name="random", line=dict(color=viz.ink("light")[0], width=1.5, dash="dot")))
fig.update_layout(xaxis_title="clientes tratados", yaxis_title="ganho incremental acumulado")
fig

## 5. Variante — propensity **fixa por conjunto de canais** de envio

Meio-termo entre a fixa global (um número) e a estimada (contínua por X). A §3
mostrou que o eixo dominante do `model_p` é **canal** (`n_channels`,
`channel_social`) — o que faz sentido: ver depende de *como* a oferta chegou.
Esta variante fixa `g` na **taxa de view empírica observada em cada combinação
de canais**, por `offer_type`, e aplica esse valor a todas as linhas do grupo —
sem deixar features de perfil de cliente entrarem no peso.

É propensity *conhecida por estrato de entrega*, não *estimada por covariável de
cliente*: reconhece a autosseleção-por-canal (observável e ligada ao envio, não
ao cliente) e ignora o resto. Como no passo 2/4, o X-learner é o mesmo — só o `p`
muda, via `model.predict(p=...)`.

In [18]:
CHANNEL_FLAGS = ["channel_web", "channel_email", "channel_mobile", "channel_social"]

def channel_key(df):
    """Chave do conjunto de canais: a tupla das 4 flags como string (ex. '1010')."""
    return df[CHANNEL_FLAGS].astype(int).astype(str).agg("".join, axis=1)

# Taxa de view por (offer_type, conjunto de canais) — a propensity fixa por estrato.
holdout_df = holdout_df.copy()
holdout_df["_chan"] = channel_key(holdout_df)

view_por_canal = (
    holdout_df.groupby([OFFER_TYPE_COLUMN, "_chan"])
    .agg(view_rate=(TREATMENT_COLUMN, "mean"), n=(TREATMENT_COLUMN, "size"))
    .reset_index()
    .sort_values([OFFER_TYPE_COLUMN, "n"], ascending=[True, False])
)
print(f"{view_por_canal.shape[0]} estratos (offer_type × conjunto de canais)")
view_por_canal

6 estratos (offer_type × conjunto de canais)


,offer_type,_chan,view_rate,n
2,bogo,1111,0.894210,5095
1,bogo,1110,0.490916,2587
0,bogo,0111,0.797940,2524
5,discount,1111,0.853408,5164
3,discount,1100,0.301775,2535
4,discount,1110,0.481452,2507


In [19]:
def propensity_fixed_by_channel(df, cfg):
    """p fixo por (offer_type, conjunto de canais) = taxa de view do estrato.
    Guard p∈(0,1) aberto (CausalML rejeita 0/1): estrato degenerado ou pequeno
    (< min_n) cai para a taxa de view global do offer_type."""
    min_n = 30  # estratos com menos linhas usam a taxa global do tipo (ad-hoc)
    p_out = pd.Series(index=df.index, dtype=float)
    for offer_type, group in df.groupby(OFFER_TYPE_COLUMN):
        global_rate = float(group[TREATMENT_COLUMN].mean())
        rates = group.groupby("_chan")[TREATMENT_COLUMN].agg(["mean", "size"])
        for chan, sub_idx in group.groupby("_chan").groups.items():
            r, n = rates.loc[chan, "mean"], rates.loc[chan, "size"]
            if n < min_n or r <= 0.0 or r >= 1.0:
                r = global_rate
            r = min(max(r, 1e-3), 1 - 1e-3)  # guard final p∈(0,1)
            p_out.loc[sub_idx] = r
    return p_out

p_channel = propensity_fixed_by_channel(holdout_df, cfg)
uplift_channel = predict_uplift_with_p(models, holdout_df, p_channel)
qini_channel = uplift_eval.qini(holdout_df["converted"], uplift_channel, holdout_df["treatment"])
print(f"Qini AUC (propensity fixa por conjunto de canais) = {qini_channel:.4f}")
print(f"valores distintos de p: {p_channel.nunique()}  (vs. contínua da produção)")

Qini AUC (propensity fixa por conjunto de canais) = 0.0195
valores distintos de p: 6  (vs. contínua da produção)


### Referência — propensity **fixa global** (a versão anterior à mudança de 2026-07-16)

Para fechar o páreo: a fixa global é `p = taxa de view` constante, por `offer_type`.
Sob a fórmula do X-learner, um `p` constante some da *ordenação* (afeta o nível de
τ, não o rank), então é o piso de "propensity que não contamina o ranking".

In [20]:
def propensity_fixed_global(df):
    p_out = pd.Series(index=df.index, dtype=float)
    for offer_type, group in df.groupby(OFFER_TYPE_COLUMN):
        r = float(group[TREATMENT_COLUMN].mean())
        p_out.loc[group.index] = min(max(r, 1e-3), 1 - 1e-3)
    return p_out

p_global = propensity_fixed_global(holdout_df)
uplift_global = predict_uplift_with_p(models, holdout_df, p_global)
qini_global = uplift_eval.qini(holdout_df["converted"], uplift_global, holdout_df["treatment"])
print(f"Qini AUC (propensity fixa global) = {qini_global:.4f}")

Qini AUC (propensity fixa global) = 0.0335


### Placar completo — cinco variantes de propensity, mesmo X-learner

In [21]:
placar_completo = pd.DataFrame({
    "variante": [
        "fixa global (pré-2026-07-16)",
        "fixa por conjunto de canais",
        "estimada — sem features compartilhadas",
        "estimada — clipada [0.1, 0.9]",
        "estimada — todas as features (produção)",
    ],
    "qini_auc": [qini_global, qini_channel, qini_reduced, qini_clip, qini_base],
}).sort_values("qini_auc", ascending=False).reset_index(drop=True)
placar_completo

,variante,qini_auc
0,fixa global (pré-2026-07-16),0.033539
1,fixa por conjunto de canais,0.019502
2,"estimada — clipada [0.1, 0.9]",0.017981
3,estimada — sem features compartilhadas,0.014732
4,estimada — todas as features (produção),0.013207


In [22]:
# Barras horizontais do placar — quanto cada variante recupera vs. a produção.
ordem = placar_completo.sort_values("qini_auc")  # menor embaixo
fig = viz.horizontal_bars(
    ordem, category="variante", value="qini_auc",
    title="Qini por variante de propensity sobre o mesmo X-learner",
    subtitle="Holdout real (bogo+discount, sem informational). Só o peso g(x) muda; os estimadores de CATE são idênticos.",
    xlabel="Qini AUC",
    labels=False,
)
fig

In [23]:
# Curvas Qini de todas as variantes.
import plotly.graph_objects as go
pal = viz.palette("light")

variantes = [
    (uplift_global,  f"fixa global ({qini_global:.3f})"),
    (uplift_channel, f"fixa por canais ({qini_channel:.3f})"),
    (uplift_reduced, f"sem compartilhadas ({qini_reduced:.3f})"),
    (uplift_clip,    f"clipada [0.1,0.9] ({qini_clip:.3f})"),
    (uplift_base,    f"estimada/todas ({qini_base:.3f})"),
]
fig = viz.figure(
    title="Curvas Qini — cinco variantes de propensity",
    subtitle="Ganho incremental acumulado no holdout, ordenado por uplift previsto.",
)
ultima = None
for i, (up, nome) in enumerate(variantes):
    curva = uplift_eval.qini_curve_points(holdout_df["converted"], up, holdout_df["treatment"])
    ultima = curva
    fig.add_trace(go.Scatter(x=curva["n_treated"], y=curva["gain"], mode="lines",
                             name=nome, line=dict(color=pal[i % len(pal)], width=2.5)))
# random = reta entre as pontas (mesma para todas: depende só de N e do total de conversões).
fig.add_trace(go.Scatter(
    x=[ultima["n_treated"].iloc[0], ultima["n_treated"].iloc[-1]],
    y=[ultima["gain"].iloc[0], ultima["gain"].iloc[-1]],
    mode="lines", name="random", line=dict(color=viz.ink("light")[0], width=1.5, dash="dot")))
fig.update_layout(xaxis_title="clientes tratados", yaxis_title="ganho incremental acumulado")
fig

### Fechamento — `g(x)` mean/median/std por variante

A tese do estudo num placar só: **quanto menos `g(x)` varia entre clientes,
maior o Qini**. Cada variante entra com a estatística do seu vetor de propensity
(a média e a mediana localizam o valor; o **std** é o que conta a história — é a
dispersão de `g(x)` entre clientes que reintroduz um eixo não-causal no ranking
do X-learner).

In [24]:
# Reconstrói os p das duas variantes estimadas (mesmo model_p já ajustado / clip).
p_base = pd.Series(index=holdout_df.index, dtype=float)
for offer_type, group in holdout_df.groupby(OFFER_TYPE_COLUMN):
    X = _design_matrix(group)
    p_base.loc[group.index] = models[offer_type].model_p.predict_proba(X)[:, 1]
p_clip = p_base.clip(0.1, 0.9)

vetores = {
    "fixa global (pré-2026-07-16)":            (p_global,  qini_global),
    "fixa por conjunto de canais":             (p_channel, qini_channel),
    "estimada — clipada [0.1, 0.9]":           (p_clip,    qini_clip),
    "estimada — sem features compartilhadas":  (p_reduced, qini_reduced),
    "estimada — todas as features (produção)": (p_base,    qini_base),
}

placar_gx = pd.DataFrame([
    {
        "variante": nome,
        "qini_auc": qini,
        "g(x) mean": p.mean(),
        "g(x) median": p.median(),
        "g(x) std": p.std(),
        "n valores distintos": p.round(6).nunique(),
    }
    for nome, (p, qini) in vetores.items()
]).sort_values("qini_auc", ascending=False).reset_index(drop=True)

placar_gx.style.format({
    "qini_auc": "{:.4f}", "g(x) mean": "{:.3f}",
    "g(x) median": "{:.3f}", "g(x) std": "{:.3f}",
})

,variante,qini_auc,g(x) mean,g(x) median,g(x) std,n valores distintos
0,fixa global (pré-2026-07-16),0.0335,0.697,0.697,0.072,2
1,fixa por conjunto de canais,0.0195,0.697,0.853,0.218,6
2,"estimada — clipada [0.1, 0.9]",0.0180,0.684,0.807,0.255,14294
3,estimada — sem features compartilhadas,0.0147,0.697,0.810,0.266,19894
4,estimada — todas as features (produção),0.0132,0.694,0.807,0.266,20037


In [25]:
# g(x) std vs. Qini — o eixo que ordena o placar inteiro.
import plotly.graph_objects as go
pal = viz.palette("light")

fig = viz.figure(
    title="Qini cai com a dispersão de g(x) — propensity fixa (std=0) é a de maior Qini",
    subtitle="Cada ponto é uma variante. Std de g(x) entre clientes no eixo x; Qini AUC no eixo y.",
)
fig.add_trace(go.Scatter(
    x=placar_gx["g(x) std"], y=placar_gx["qini_auc"],
    mode="markers+text", text=placar_gx["variante"], textposition="top center",
    textfont=dict(size=9), marker=dict(color=pal[1], size=11),
    hovertemplate="%{text}<br>g(x) std=%{x:.3f}<br>Qini=%{y:.4f}<extra></extra>",
))
fig.update_layout(xaxis_title="std de g(x) entre clientes", yaxis_title="Qini AUC", showlegend=False)
fig